# Case Study A: Comparative Analysis of Classification Algorithms for Bank Marketing Prediction
**Course:** Fundamentals of Data Science | **Student:** Raghav Sharma (160124733196)

## Task 1 - Data understanding and preprocessing
Load the semicolon-delimited CSV and inspect its structure.

In [ ]:
import warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             confusion_matrix, ConfusionMatrixDisplay, roc_curve, roc_auc_score)

sns.set_theme(style="whitegrid")
RANDOM_STATE = 42
DATA_PATH = "bank-additional-full.csv"   # change if your file has a different name

df = pd.read_csv(DATA_PATH, sep=";")      # the file uses ';' as delimiter
display(df.head(10))
print("Dimensions (rows, columns):", df.shape)

### 1.1 Data types and descriptive statistics
Purpose: confirm which columns are numeric or categorical and see the scale/spread of numeric columns.

In [ ]:
print(df.dtypes)
display(df.describe())
display(df.describe(include="object"))

### 1.2 Attribute types, unknown entries, duplicates and target distribution
In this dataset missing values are stored as the text `unknown`, so `isnull()` alone would miss them.

In [ ]:
target = "y"
num_cols = df.drop(columns=target).select_dtypes(include="number").columns.tolist()
cat_cols = df.drop(columns=target).select_dtypes(exclude="number").columns.tolist()
print("Numerical attributes :", num_cols)
print("Categorical attributes:", cat_cols)
print("Target attribute      :", target)

print("\nTrue null values:", int(df.isnull().sum().sum()))
unknown_counts = (df[cat_cols] == "unknown").sum()
print("\n'unknown' entries per categorical column:")
print(unknown_counts[unknown_counts > 0])

print("\nDuplicate records:", int(df.duplicated().sum()))
print("\nTarget distribution:")
print(df[target].value_counts())
print((df[target].value_counts(normalize=True) * 100).round(2))

### 1.3 Preprocessing, encoding, scaling and 80:20 split
Decisions made here (state these in the report):
* `duration` is **dropped** - it is only known after the call ends, so it cannot be used for pre-contact prediction.
* Duplicate rows are removed so identical records cannot appear in both train and test sets.
* `unknown` is kept as its own category rather than guessed.
* Categorical columns are one-hot encoded; numeric columns are standardised. The transformer is fitted on the **training set only** to avoid data leakage.
* A **stratified** split keeps the same class ratio in train and test.

In [ ]:
df_model = df.drop_duplicates().drop(columns=["duration"]).copy()
df_model[target] = (df_model[target] == "yes").astype(int)

X = df_model.drop(columns=target)
y = df_model[target]
num_cols_m = [c for c in num_cols if c != "duration"]
cat_cols_m = cat_cols

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=RANDOM_STATE)

preprocessor = ColumnTransformer([
    ("num", StandardScaler(), num_cols_m),
    ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols_m),
])
Xtr = preprocessor.fit_transform(X_train)
Xte = preprocessor.transform(X_test)
print("Train:", Xtr.shape, " Test:", Xte.shape)
print("Subscription rate  train: %.2f%%   test: %.2f%%" % (y_train.mean()*100, y_test.mean()*100))

## Task 2 - Exploratory data analysis
### 2.1 Target class distribution

In [ ]:
counts = y.value_counts().rename({0: "No", 1: "Yes"})
pct = (counts / counts.sum() * 100).round(2)
print(pd.DataFrame({"Customers": counts, "Percentage": pct}))

ax = counts.plot(kind="bar", color=["#c0504d", "#4f81bd"], rot=0, figsize=(5, 4))
for i, v in enumerate(counts):
    ax.text(i, v, f"{v}\n({pct.iloc[i]}%)", ha="center", va="bottom")
ax.set_title("Distribution of Term-Deposit Subscription (Target)")
ax.set_xlabel("Subscribed to term deposit?")
ax.set_ylabel("Number of customers")
plt.tight_layout(); plt.show()

### 2.2 Subscription rate across customer and campaign attributes

In [ ]:
eda = df_model.copy()
eda["y_pct"] = eda[target] * 100
cols = [c for c in ["job", "month", "poutcome", "contact"] if c in eda.columns]
fig, axes = plt.subplots(2, 2, figsize=(13, 9))
for ax, c in zip(axes.ravel(), cols):
    rate = eda.groupby(c)["y_pct"].mean().sort_values(ascending=False)
    sns.barplot(x=rate.values, y=rate.index, ax=ax, color="#4f81bd")
    ax.axvline(eda["y_pct"].mean(), color="red", ls="--", label="Overall rate")
    ax.set_title(f"Subscription Rate by {c}")
    ax.set_xlabel("Subscription rate (%)"); ax.set_ylabel(c); ax.legend()
plt.tight_layout(); plt.show()

### 2.3 Numerical variables vs subscription outcome

In [ ]:
nums = [c for c in ["age", "campaign", "previous", "euribor3m"] if c in df_model.columns]
fig, axes = plt.subplots(1, len(nums), figsize=(4.2 * len(nums), 4))
for ax, c in zip(np.atleast_1d(axes), nums):
    sns.boxplot(data=df_model, x=target, y=c, ax=ax, palette=["#c0504d", "#4f81bd"])
    ax.set_xticklabels(["No", "Yes"])
    ax.set_title(f"{c} by Subscription Outcome")
    ax.set_xlabel("Subscribed?"); ax.set_ylabel(c)
plt.tight_layout(); plt.show()
display(df_model.groupby(target)[nums].mean().round(2).rename(index={0: "No", 1: "Yes"}))

## Task 3 - Classification model development
Same split, same features, same target for all four models.

In [ ]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=RANDOM_STATE),
    "Decision Tree": DecisionTreeClassifier(max_depth=6, random_state=RANDOM_STATE),
    "KNN (k=5)": KNeighborsClassifier(n_neighbors=5),
    "Random Forest": RandomForestClassifier(n_estimators=200, random_state=RANDOM_STATE, n_jobs=-1),
}
rows, preds, probas = [], {}, {}
for name, m in models.items():
    m.fit(Xtr, y_train)
    p = m.predict(Xte)
    s = m.predict_proba(Xte)[:, 1]
    preds[name], probas[name] = p, s
    rows.append({"Model": name,
                 "Accuracy": accuracy_score(y_test, p),
                 "Precision": precision_score(y_test, p, zero_division=0),
                 "Recall": recall_score(y_test, p),
                 "F1-score": f1_score(y_test, p),
                 "ROC-AUC": roc_auc_score(y_test, s)})
results = pd.DataFrame(rows).set_index("Model").round(4)
display(results)

## Task 4 - Comparison and visualisation
### 4.1 Grouped bar chart of metrics

In [ ]:
ax = results[["Accuracy", "Precision", "Recall", "F1-score"]].plot(kind="bar", figsize=(10, 5), rot=0, width=0.8)
ax.set_title("Model Comparison: Accuracy, Precision, Recall and F1-score")
ax.set_xlabel("Classifier"); ax.set_ylabel("Score"); ax.set_ylim(0, 1.05)
for cont in ax.containers:
    ax.bar_label(cont, fmt="%.2f", fontsize=8)
ax.legend(loc="upper right", ncol=4)
plt.tight_layout(); plt.show()

### 4.2 Confusion matrices

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(10, 8))
for ax, (name, p) in zip(axes.ravel(), preds.items()):
    ConfusionMatrixDisplay(confusion_matrix(y_test, p), display_labels=["No", "Yes"]).plot(ax=ax, cmap="Blues", colorbar=False)
    ax.set_title(f"Confusion Matrix - {name}")
    ax.set_xlabel("Predicted label"); ax.set_ylabel("True label")
plt.tight_layout(); plt.show()

cm_rows = []
for name, p in preds.items():
    tn, fp, fn, tp = confusion_matrix(y_test, p).ravel()
    cm_rows.append({"Model": name, "TN": tn, "FP": fp, "FN": fn, "TP": tp})
display(pd.DataFrame(cm_rows).set_index("Model"))

### 4.3 ROC curves and ROC-AUC

In [ ]:
plt.figure(figsize=(7, 6))
for name, s in probas.items():
    fpr, tpr, _ = roc_curve(y_test, s)
    plt.plot(fpr, tpr, label=f"{name} (AUC = {roc_auc_score(y_test, s):.3f})")
plt.plot([0, 1], [0, 1], "k--", label="Random guess")
plt.title("ROC Curves of the Four Classifiers")
plt.xlabel("False positive rate"); plt.ylabel("True positive rate (recall)")
plt.legend(loc="lower right"); plt.tight_layout(); plt.show()

### 4.4 Ability to identify actual subscribers and best-model selection
Accuracy alone is not used because of class imbalance. Models are ranked on ROC-AUC and F1, with recall inspected for the business cost of missed subscribers.

In [ ]:
n_pos = int(y_test.sum())
ident = pd.DataFrame({
    "Actual subscribers in test set": n_pos,
    "Correctly identified (TP)": {n: int(((preds[n] == 1) & (y_test == 1)).sum()) for n in preds},
    "Missed (FN)": {n: int(((preds[n] == 0) & (y_test == 1)).sum()) for n in preds},
    "Non-subscribers wrongly called (FP)": {n: int(((preds[n] == 1) & (y_test == 0)).sum()) for n in preds},
})
display(ident)

rank = results[["Precision", "Recall", "F1-score", "ROC-AUC"]].rank(ascending=False)
rank["Average rank"] = rank.mean(axis=1)
display(rank.sort_values("Average rank"))
print("Best by ROC-AUC :", results["ROC-AUC"].idxmax())
print("Best by F1-score:", results["F1-score"].idxmax())
print("Best by Recall  :", results["Recall"].idxmax())

### 4.5 Supplementary: effect of class weighting (optional)
Default models usually miss many subscribers. Re-training with `class_weight="balanced"` shows the recall/precision trade-off. KNN has no such option and is omitted.

In [ ]:
bal = {
    "Logistic Regression": LogisticRegression(max_iter=1000, class_weight="balanced", random_state=RANDOM_STATE),
    "Decision Tree": DecisionTreeClassifier(max_depth=6, class_weight="balanced", random_state=RANDOM_STATE),
    "Random Forest": RandomForestClassifier(n_estimators=200, class_weight="balanced_subsample", random_state=RANDOM_STATE, n_jobs=-1),
}
rows = []
for name, m in bal.items():
    m.fit(Xtr, y_train); p = m.predict(Xte); s = m.predict_proba(Xte)[:, 1]
    rows.append({"Model": name + " (balanced)", "Accuracy": accuracy_score(y_test, p),
                 "Precision": precision_score(y_test, p, zero_division=0), "Recall": recall_score(y_test, p),
                 "F1-score": f1_score(y_test, p), "ROC-AUC": roc_auc_score(y_test, s)})
display(pd.DataFrame(rows).set_index("Model").round(4))

## Task 5 - Interpretation
Top drivers of subscription according to the Random Forest (supports campaign-targeting recommendations).

In [ ]:
rf = models["Random Forest"]
names = preprocessor.get_feature_names_out()
imp = pd.Series(rf.feature_importances_, index=names).sort_values(ascending=False).head(10)[::-1]
imp.plot(kind="barh", figsize=(8, 5), color="#4f81bd")
plt.title("Top 10 Feature Importances - Random Forest")
plt.xlabel("Importance"); plt.ylabel("Feature")
plt.tight_layout(); plt.show()